# 03_02 - Mapa integrado SER + EMT/off-street

Ejecución fina del mapa reusable definido en `src.visualization.parking_map`.
El HTML se genera en `reports/maps/` y no se incrusta en el notebook.

In [10]:
from pathlib import Path
import sys

import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 160)


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current] + list(current.parents):
        if (candidate / "data_catalog.csv").exists():
            return candidate
    raise FileNotFoundError("No se ha encontrado data_catalog.csv.")


ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

ROOT

PosixPath('/Users/hugo/TFM_parking_madrid')

In [11]:
from src.visualization.parking_map import build_ser_emt_map_from_paths

result = build_ser_emt_map_from_paths(
    root=ROOT,
    target_year=2026,
    html_output_path=Path("reports/maps/mapa_ser_emt_integrado.html"),
    png_output_path=Path("reports/figures/mapa_ser_emt_integrado.png"),
    barrios_model_png_output_path=Path("reports/figures/mapa_ser_barrios_modelo.png"),
)

result.outputs

,output,path,exists,size_mb
0,html_integrado,reports/maps/mapa_ser_emt_integrado.html,True,19.512
1,png_integrado,reports/figures/mapa_ser_emt_integrado.png,True,5.117
2,png_barrios_modelo,reports/figures/mapa_ser_barrios_modelo.png,True,3.259


## Checks

In [12]:
result.checks

,check_id,status,detail,critical
0,ser_layers_exist,OK,ser_geoportal_limite_ser=data/interim/cartografia/ser_geoportal_limite_ser/ser_geoportal_limite_ser_clean.parquet:True; ser_geoportal_barrios_ser=data/inter...,True
1,ser_layers_not_empty,OK,ser_geoportal_limite_ser:rows=1; ser_geoportal_barrios_ser:rows=66; ser_geoportal_bandas_aparcamiento:rows=34450; callejero_viales_vigentes:rows=9287; ser_p...,True
2,ser_crs_epsg_25830,OK,ser_geoportal_limite_ser:epsg=25830; ser_geoportal_barrios_ser:epsg=25830; ser_geoportal_bandas_aparcamiento:epsg=25830; callejero_viales_vigentes:epsg=2583...,True
3,emt_file_exists,OK,/Users/hugo/TFM_parking_madrid/data/processed/core/emt/inventario_global_emt.parquet,True
4,emt_not_empty,OK,rows=85,True
5,emt_expected_entities,OK,rows=85; expected=85,True
6,emt_required_columns,OK,missing=[],True
7,emt_parking_uid_not_null,OK,nulls=0,True
8,emt_parking_uid_unique,OK,duplicates=0,True
9,emt_coordinates_valid,OK,"lat_nulls=0; lon_nulls=0; lat_range=[40.36521668890186, 40.4941929509737]; lon_range=[-3.78414489510692, -3.598831]",True


## Compatibilidad barrios mapa-modelo

In [13]:
result.diagnostics["barrios_model_compatibility"]

,metric,value
0,filas_cartografia_original,66
1,barrio_key_unicos_cartografia,65
2,filas_tras_disolver,65
3,barrios_modelo,65
4,duplicados_originales_por_barrio_key,1
5,keys_cartografia_no_modelo,0
6,keys_modelo_no_cartografia,0
7,caso_09_04,Valdezarza / Valdezarza Fase III
8,expected_model_barrios,65


In [14]:
result.diagnostics["barrios_duplicate_keys"]

,barrio_key,n_geometrias_origen,nombres_cartograficos,cod_barrio
48,09_04,2,Valdezarza / Valdezarza Fase III,904


## Diagnóstico EMT

In [15]:
result.diagnostics["emt_inventory"]

,n_rows,n_parking_uid,crs_epsg
0,85,85,25830


In [16]:
result.diagnostics["emt_spatial_filter"]

,n_emt_inventory_total,n_emt_in_visual_area,n_emt_outside_visual_area,visual_buffer_m
0,85,74,11,25


## Plazas SER

In [17]:
result.diagnostics["ser_plazas"]

,total_numero_plazas_bandas_ser,total_plazas_barrio_anio_target,diferencia_absoluta,diferencia_relativa,status
0,181005.0,181079.0,74.0,0.000409,OK


## Outputs

In [18]:
result.outputs

,output,path,exists,size_mb
0,html_integrado,reports/maps/mapa_ser_emt_integrado.html,True,19.512
1,png_integrado,reports/figures/mapa_ser_emt_integrado.png,True,5.117
2,png_barrios_modelo,reports/figures/mapa_ser_barrios_modelo.png,True,3.259
